# Market Basket Analysis

Market basket analysis scrutinizes the products customers tend to buy together, and uses the information to decide which products should be cross-sold or promoted together. The term arises from the shopping carts supermarket shoppers fill up during a shopping trip.

Association Rule Mining is used when we want to find an association between different objects in a set, find frequent patterns in a transaction database, relational databases or any other information repository.

The most common approach to find these patterns is Market Basket Analysis, which is a key technique used by large retailers like Amazon, Flipkart, etc to analyze customer buying habits by finding associations between the different items that customers place in their “shopping baskets”. The discovery of these associations can help retailers develop marketing strategies by gaining insight into which items are frequently purchased together by customers. The strategies may include:

- Changing the store layout according to trends
- Customers behavior analysis
- Catalog Design
- Cross marketing on online stores
- Customized emails with add-on sales, etc.

### Matrices

- **Support** : Its the default popularity of an item. In mathematical terms, the support of item A is the ratio of transactions involving A to the total number of transactions.


- **Confidence** : Likelihood that customer who bought both A and B. It is the ratio of the number of transactions involving both A and B and the number of transactions involving B.
     - Confidence(A => B) = Support(A, B)/Support(A)


- **Lift** : Increase in the sale of A when you sell B.
    
    - Lift(A => B) = Confidence(A, B)/Support(B)
        
    - Lift (A => B) = 1 means that there is no correlation within the itemset.
    - Lift (A => B) > 1 means that there is a positive correlation within the itemset, i.e., products in the itemset, A, and B, are more likely to be bought together.
    - Lift (A => B) < 1 means that there is a negative correlation within the itemset, i.e., products in itemset, A, and B, are unlikely to be bought together.

**Apriori Algorithm:** Apriori algorithm assumes that any subset of a frequent itemset must be frequent. Its the algorithm behind Market Basket Analysis. Say, a transaction containing {Grapes, Apple, Mango} also contains {Grapes, Mango}. So, according to the principle of Apriori, if {Grapes, Apple, Mango} is frequent, then {Grapes, Mango} must also be frequent.

In [1]:
import numpy as np
import pandas as pd


root = r"C:\Users\MHMD RAGAB\Downloads\DATA\New folder\\"

### Data

In [2]:
order_products__prior = pd.read_csv(r'C:\Users\MHMD RAGAB\Downloads\DATA\New folder\order_products__prior.csv')
order_products__train = pd.read_csv(r'C:\Users\MHMD RAGAB\Downloads\DATA\New folder\order_products__train.csv')
orders = pd.read_csv(r'C:\Users\MHMD RAGAB\Downloads\DATA\New folder\orders.csv')
products = pd.read_csv(r'C:\Users\MHMD RAGAB\Downloads\DATA\New folder\products.csv')

In [6]:
order_products = pd.concat(
    [order_products__prior, order_products__train],
    ignore_index=True
)

In [7]:
order_products.head()

,order_id,product_id,add_to_cart_order,reordered
0,2,33120,1,1
1,2,28985,2,1
2,2,9327,3,0
3,2,45918,4,1
4,2,30035,5,0


In [8]:
order_products.product_id.nunique()

49685

Out of 49685 keeping top 100 most frequent products.

In [9]:
product_counts = order_products.groupby('product_id')['order_id'].count().reset_index().rename(columns = {'order_id':'frequency'})
product_counts = product_counts.sort_values('frequency', ascending=False)[0:100].reset_index(drop = True)
product_counts = product_counts.merge(products, on = 'product_id', how = 'left')
product_counts.head(10)

,product_id,frequency,product_name,aisle_id,department_id
0,24852,491291,Banana,24,4
1,13176,394930,Bag of Organic Bananas,24,4
2,21137,275577,Organic Strawberries,24,4
3,21903,251705,Organic Baby Spinach,123,4
4,47209,220877,Organic Hass Avocado,24,4
5,47766,184224,Organic Avocado,24,4
6,47626,160792,Large Lemon,24,4
7,16797,149445,Strawberries,24,4
8,26209,146660,Limes,24,4
9,27845,142813,Organic Whole Milk,84,16


Keeping 100 most frequent items in order_products dataframe

In [10]:
freq_products = list(product_counts.product_id)
freq_products[1:10]

[13176, 21137, 21903, 47209, 47766, 47626, 16797, 26209, 27845]

In [11]:
len(freq_products)

100

In [12]:
order_products = order_products[order_products.product_id.isin(freq_products)]
order_products.shape

(7795471, 4)

In [13]:
order_products.order_id.nunique()

2444982

In [14]:
order_products = order_products.merge(products, on = 'product_id', how='left')
order_products.head()

,order_id,product_id,add_to_cart_order,reordered,product_name,aisle_id,department_id
0,2,28985,2,1,Michigan Organic Kale,83,4
1,2,17794,6,1,Carrots,83,4
2,3,24838,2,1,Unsweetened Almondmilk,91,16
3,3,21903,4,1,Organic Baby Spinach,123,4
4,3,46667,6,1,Organic Ginger Root,83,4


Structuring the data for feeding in the algorithm

In [15]:
basket = order_products.groupby(['order_id', 'product_name'])['reordered'].count().unstack().reset_index().fillna(0).set_index('order_id')
basket.head()

product_name,100% Raw Coconut Water,100% Whole Wheat Bread,2% Reduced Fat Milk,Apple Honeycrisp Organic,Asparagus,Bag of Organic Bananas,Banana,Bartlett Pears,Blueberries,Boneless Skinless Chicken Breasts,...,Sparkling Natural Mineral Water,Sparkling Water Grapefruit,Spring Water,Strawberries,Uncured Genoa Salami,Unsalted Butter,Unsweetened Almondmilk,Unsweetened Original Almond Breeze Almond Milk,Whole Milk,Yellow Onions
order_id,,,,,,,,,,,,,,,,,,,,,
1,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0
5,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
9,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [20]:
import gc
gc.collect()

2491

encoding the units

In [22]:
basket = (basket > 0).astype(int)

In [23]:
basket.size

244498200

In [24]:
basket.shape

(2444982, 100)

Creating frequent sets and rules

In [27]:
!pip install mlxtend

INFO: pip is looking at multiple versions of mlxtend to determine which version is compatible with other requirements. This could take a while.
   ---------------------------------------- 0.0/1.4 MB ? eta -:--:--
   ------- -------------------------------- 0.3/1.4 MB ? eta -:--:--
   ------------------------------- -------- 1.0/1.4 MB 4.2 MB/s eta 0:00:01
   ---------------------------------------- 1.4/1.4 MB 4.1 MB/s eta 0:00:00


In [30]:
gc.collect()

2375

In [32]:
basket_small = basket.sample(n=5000, random_state=42)

frequent_items = apriori(
    basket_small,
    min_support=0.02,
    use_colnames=True
)

C:\Users\MHMD RAGAB\anaconda3\envs\myenv_cv\lib\site-packages\mlxtend\frequent_patterns\fpcommon.py:161: DeprecationWarning: DataFrames with non-bool types result in worse computationalperformance and their support might be discontinued in the future.Please use a DataFrame with bool type
  warnings.warn(


In [33]:
# from mlxtend.frequent_patterns import apriori

# frequent_items = apriori(
#     basket,
#     min_support=0.01,
#     use_colnames=True
# )

# frequent_items.head()

In [34]:
frequent_items.tail()

,support,itemsets
63,0.0224,"(Organic Baby Spinach, Bag of Organic Bananas)"
64,0.0234,"(Organic Hass Avocado, Bag of Organic Bananas)"
65,0.0230,"(Organic Strawberries, Bag of Organic Bananas)"
66,0.0232,"(Banana, Organic Baby Spinach)"
67,0.0250,"(Banana, Organic Strawberries)"


In [35]:
frequent_items.shape

(68, 2)

In [37]:
from mlxtend.frequent_patterns import association_rules

rules = association_rules(
    frequent_items,
    metric="lift",
    min_threshold=1
)

rules.sort_values('lift', ascending=False).head()

,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,representativity,leverage,conviction,zhangs_metric,jaccard,certainty,kulczynski
2,(Organic Hass Avocado),(Bag of Organic Bananas),0.0896,0.1630,0.0234,0.261161,1.602213,1.0,0.008795,1.132858,0.412855,0.102094,0.117277,0.202359
3,(Bag of Organic Bananas),(Organic Hass Avocado),0.1630,0.0896,0.0234,0.143558,1.602213,1.0,0.008795,1.063003,0.449060,0.102094,0.059269,0.202359
0,(Organic Baby Spinach),(Bag of Organic Bananas),0.1012,0.1630,0.0224,0.221344,1.357938,1.0,0.005904,1.074929,0.293268,0.092639,0.069706,0.179384
1,(Bag of Organic Bananas),(Organic Baby Spinach),0.1630,0.1012,0.0224,0.137423,1.357938,1.0,0.005904,1.041994,0.314921,0.092639,0.040302,0.179384
4,(Organic Strawberries),(Bag of Organic Bananas),0.1114,0.1630,0.0230,0.206463,1.266645,1.0,0.004842,1.054771,0.236904,0.091488,0.051927,0.173784
